In [12]:
import numpy as np
import pandas as pd

def sort_best_window_config(results_csv):
    # 1. Load the CSV file (Replace 'your_results.csv' with your actual filename)
    csv_file_path = results_csv
    df = pd.read_csv(csv_file_path)

    # 2. Handle the gap: penalize large positive gaps (overfitting),
    # while treating negative gaps gently (good generalization)
    df["gap_penalty"] = df["final_gap"].apply(lambda x: x if x > 0 else abs(x) * 0.2)

    # 3. Define weights based on your priorities (Must sum to 1.0)
    # We place high importance on raw accuracy, but heavily penalize catastrophic upticks or gaps.
    W_ACCURACY = 0.50  # Weight for best_val_mse
    W_STABILITY = 0.30  # Weight for val_uptick_from_best
    W_GENERALIZATION = 0.20  # Weight for the gap_penalty

    # 4. Normalize the metrics to a 0-1 scale so weights apply fairly
    # (0 = best observed in the dataset, 1 = worst observed)
    def min_max_scale(series):
        if series.max() == series.min():
            return 0
        return (series - series.min()) / (series.max() - series.min())


    norm_val = min_max_scale(df["best_val_mse"])
    norm_uptick = min_max_scale(df["val_uptick_from_best"])
    norm_gap = min_max_scale(df["gap_penalty"])

    # 5. Calculate the Composite Penalty Score (Lower is better)
    df["composite_score"] = (
        (W_ACCURACY * norm_val)
        + (W_STABILITY * norm_uptick)
        + (W_GENERALIZATION * norm_gap)
    )

    # 6. Sort by the composite score
    ranked_df = df.sort_values(by="composite_score", ascending=True).reset_index(
        drop=True
    )

    # Display the top results
    print("--- RANKED BY COMPOSITE PENALTY SCORE (LOWER IS BETTER) ---")
    display(
        ranked_df[
            [
                "window_sec",
                "overlap_sec",
                "n_available",
                "n_used",
                "hop_sec",
                "composite_score",
                "best_val_mse",
                "val_uptick_from_best",
                "final_gap",
            ]
        ].head(3)
    )

In [14]:
results_csvs = [
    "pickles/encoding_2026-07-02/run_1/mini_windowtest_traincurves_summary_Sub2_acoustic_1.csv",
    "pickles/encoding_2026-07-02/run_2/mini_windowtest_traincurves_summary_Sub2_acoustic.csv",
    "pickles/encoding_2026-07-02/run_3/mini_windowtest_traincurves_summary_Sub2_acoustic.csv",
    "pickles/encoding_2026-07-02/run_4/mini_windowtest_traincurves_summary_Sub2_acoustic.csv",
    "pickles/encoding_2026-07-02/mini_windowtest_traincurves_summary_Sub2_acoustic.csv"]

for results_csv in results_csvs:
    sort_best_window_config(results_csv=results_csv)

--- RANKED BY COMPOSITE PENALTY SCORE (LOWER IS BETTER) ---


,window_sec,overlap_sec,n_available,n_used,hop_sec,composite_score,best_val_mse,val_uptick_from_best,final_gap
0,5.0,0.7,1101,100,4.3,0.079946,0.442163,0.064920,0.074358
1,10.0,1.0,519,100,9.0,0.131294,0.481224,0.063782,0.079463
2,8.0,0.7,642,100,7.3,0.139723,0.501953,0.031172,0.097146


--- RANKED BY COMPOSITE PENALTY SCORE (LOWER IS BETTER) ---


,window_sec,overlap_sec,n_available,n_used,hop_sec,composite_score,best_val_mse,val_uptick_from_best,final_gap
0,8.0,0.7,642,100,7.3,0.027058,0.563607,0.001713,0.088106
1,7.0,0.7,747,100,6.3,0.061596,0.580478,0.019053,0.121753
2,8.0,1.0,669,100,7.0,0.065270,0.632429,0.013449,0.216132


--- RANKED BY COMPOSITE PENALTY SCORE (LOWER IS BETTER) ---


,window_sec,overlap_sec,n_available,n_used,hop_sec,composite_score,best_val_mse,val_uptick_from_best,final_gap
0,6.0,1.0,936,300,5.0,0.003972,0.423063,0.001367,-0.075548
1,8.0,0.7,642,300,7.3,0.024566,0.433764,0.004047,-0.015298
2,6.0,0.7,888,300,5.3,0.064739,0.443430,0.009385,-0.028317


--- RANKED BY COMPOSITE PENALTY SCORE (LOWER IS BETTER) ---


,window_sec,overlap_sec,n_available,n_used,hop_sec,composite_score,best_val_mse,val_uptick_from_best,final_gap
0,7.0,0.5,720,600,6.5,0.069169,0.454078,0.004843,-0.040951
1,8.0,1.0,669,600,7.0,0.122858,0.496304,0.000000,-0.015454
2,6.0,0.3,825,600,5.7,0.123300,0.471539,0.004794,-0.061939


--- RANKED BY COMPOSITE PENALTY SCORE (LOWER IS BETTER) ---


,window_sec,overlap_sec,n_available,n_used,hop_sec,composite_score,best_val_mse,val_uptick_from_best,final_gap
0,5.0,4.3,6636,600,0.7,0.002978,0.450090,0.004323,0.004025
1,5.0,4.0,4674,600,1.0,0.031591,0.451712,0.014478,-0.005199
2,5.0,3.0,2343,600,2.0,1.000000,0.528999,0.147253,0.201542
